# 11 · Forest plot across specifications

Re-estimate the persistence coefficient under a battery of specifications
(dropping each control, adding 2020 church density, removing state FE, the
two-period index) and plot the coefficients with 95% CIs. Writes the figure and
a coefficient table to `../figures/`. **Manuscript: robustness figure.**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

df = pd.read_csv("../data/cleaned_data_v3.csv")
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2010": "pop_2010",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989",
})

for c in ["Index_v2", "index_1890_1990", "pct_hs_1990", "pop_2010",
          "firms_2022", "income_1989_real_2023", "church_density_2020"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["Index_v2", "pct_hs_1990", "pop_2010", "firms_2022",
                       "income_1989_real_2023", "State"])
df = df[df["pop_2010"] > 0]
df["log1p_firms_2022"] = np.log1p(df["firms_2022"])
df["log_pop_2010"] = np.log(df["pop_2010"])
z = lambda x: (x - x.mean()) / x.std(ddof=0)
df["income_1989_scaled"] = z(df["income_1989_real_2023"])
df["pct_hs_1990_scaled"] = z(df["pct_hs_1990"])

def run(label, formula, var="Index_v2", data=df):
    m = smf.ols(formula, data=data).fit(cov_type="HC3")
    lo, hi = m.conf_int().loc[var].tolist()
    return {"label": label, "beta": m.params[var], "lo": lo, "hi": hi, "N": int(m.nobs)}

specs = [
    ("Baseline OLS",
     "log1p_firms_2022 ~ Index_v2 + income_1989_scaled + pct_hs_1990_scaled + log_pop_2010 + C(State)"),
    ("No Income 1989",
     "log1p_firms_2022 ~ Index_v2 + pct_hs_1990_scaled + log_pop_2010 + C(State)"),
    ("No Education 1990",
     "log1p_firms_2022 ~ Index_v2 + income_1989_scaled + log_pop_2010 + C(State)"),
    ("No log(Pop 2010)",
     "log1p_firms_2022 ~ Index_v2 + income_1989_scaled + pct_hs_1990_scaled + C(State)"),
    ("No Income & No Education",
     "log1p_firms_2022 ~ Index_v2 + log_pop_2010 + C(State)"),
    ("No State FE",
     "log1p_firms_2022 ~ Index_v2 + income_1989_scaled + pct_hs_1990_scaled + log_pop_2010"),
]
rows = [run(lab, fml) for lab, fml in specs]

if df["church_density_2020"].notna().any():
    rows.append(run("+ church_density_2020",
        "log1p_firms_2022 ~ Index_v2 + church_density_2020 + income_1989_scaled "
        "+ pct_hs_1990_scaled + log_pop_2010 + C(State)"))

if df["index_1890_1990"].notna().any():
    d2 = df.dropna(subset=["index_1890_1990"])
    rows.append(run("Two-year index (1890-1990)",
        "log1p_firms_2022 ~ index_1890_1990 + income_1989_scaled + pct_hs_1990_scaled "
        "+ log_pop_2010 + C(State)", var="index_1890_1990", data=d2))

res = pd.DataFrame(rows)
res["effect_%"] = (np.exp(res["beta"]) - 1) * 100
res.to_csv("../figures/persistence_coef_specs.csv", index=False)

# horizontal CI plot, baseline coefficient marked
res = res.iloc[::-1].reset_index(drop=True)
y = np.arange(len(res))
fig, ax = plt.subplots(figsize=(7, 5.2), dpi=200)
ax.hlines(y, res["lo"], res["hi"], lw=2)
ax.plot(res["beta"], y, "o", ms=6)
ax.axvline(float(res.loc[res["label"] == "Baseline OLS", "beta"].iloc[0]), ls="--", lw=1.5)
ax.set_yticks(y); ax.set_yticklabels(res["label"])
ax.set_xlabel("Coefficient on persistence index (95% CI)")
ax.set_title("Persistence coefficient across specifications")
ax.grid(True, axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig("../figures/coef_plot.png", bbox_inches="tight")
plt.show()